In [2]:
import pandas as pd
import numpy as np

In [3]:
customers = pd.read_csv(r'C:\Users\aveni\Day6\Datasets\customers.csv')
products = pd.read_csv(r'C:\Users\aveni\Day6\Datasets\products.csv')
orders = pd.read_csv(r'C:\Users\aveni\Day6\Datasets\orders.csv')
order_items = pd.read_csv(r'C:\Users\aveni\Day6\Datasets\order_items.csv')

In [4]:
customers_clean = customers.copy()
products_clean = products.copy()
orders_clean = orders.copy()
order_items_clean = order_items.copy()

In [5]:
customers_clean = customers_clean.drop_duplicates().copy()
products_clean = products_clean.drop_duplicates().copy()
orders_clean = orders_clean.drop_duplicates().copy()
order_items_clean = order_items_clean.drop_duplicates().copy()

In [6]:
# Clean Products 

In [7]:
products_clean["category_clean"] = (
    products_clean["category"]
    .astype("string")
    .str.strip()
    .str.lower()
)

In [8]:
category_map = {
    "electronics": "Electronics",
    "elec": "Electronics",

    "home & kitchen": "Home & Kitchen",
    "home and kitchen": "Home & Kitchen",

    "books": "Books",
    "book": "Books",

    "groceries": "Groceries",
    "grocery": "Groceries",

    "toys": "Toys",
    "toy": "Toys",

    "beauty": "Beauty",
    "beauty & care": "Beauty",

    "fashion": "Fashion",
    "fashon": "Fashion",

    "sports": "Sports",
    "sports & outdoor": "Sports",
}

In [9]:
products_clean["category_clean"] = (
    products_clean["category_clean"]
    .replace(category_map)
)


In [10]:
totCost_by_cat_clean = (
    products_clean.groupby('category_clean')['unit_cost']
        .sum()
        .sort_values(ascending=True)
)


In [12]:
products_clean["uom_clean"] = (
    products_clean["uom"]
    .astype("string")
    .str.strip()
    .str.lower()
)

In [13]:
uom_map = {
    "pc": "piece",
    "pcs": "piece",
}

In [14]:
products_clean["uom_clean"] = (
    products_clean["uom_clean"]
    .replace(uom_map)
)


In [16]:
def clean_money(series):
    return pd.to_numeric(
        series.astype("string")
        .str.replace("₱", "", regex=False)
        .str.replace(",", "", regex=False)
        .str.strip(),
        errors="coerce"
    )

In [17]:
products_clean["unit_price_clean"] = clean_money(
    products_clean["unit_price"]
)

In [18]:
products_clean["unit_price_clean"].isna().sum()

np.int64(0)

In [19]:
products_clean = products_clean[
    products_clean["unit_price_clean"].notna()
].copy()

In [40]:
# Clean Customers

In [15]:
customers_clean["gender_clean"] = (
    customers_clean["gender"]
    .astype("string")
    .str.strip()
    .str.lower()
)

In [16]:
gender_map = {
    "F": "Female",
    "f": "Female",
    "female": "Female",
    
    "M": "Male",
    "m": "Male",
    "male": "Male",

    "unknown": "Unknown",
}

In [17]:
customers_clean["gender_clean"] = (
    customers_clean["gender_clean"]
    .replace(gender_map)
)


In [18]:
customers_clean["gender_clean"] = (
    customers_clean["gender_clean"]
    .replace(r'^\s*$', "Unknown", regex=True)
    .fillna("Unknown")
)


In [19]:
customers_clean["email_clean"] = (
    customers_clean["email"]
    .astype("string")
    .str.strip()
    .str.lower()
)
customers_clean["email_clean"] = (
    customers_clean["email_clean"]
    .replace(r'^\s*$', "unknown@unknown.com", regex=True)
    .fillna("unknown@unknown.com")
)


In [31]:
customers_clean["age_clean"] = pd.to_numeric(
    customers_clean["age"],
    errors="coerce"
)

In [32]:
customers_clean.loc[
    customers_clean["age_clean"] < 0,
    "age_clean"
]=0

In [121]:
# Clean Orders

In [35]:
orders_clean["order_datetime_clean"] = pd.to_datetime(
    orders_clean["order_datetime"],
    format="mixed",
    errors="coerce"
)

In [20]:
orders_clean["discount_pct_clean"] = pd.to_numeric(
    orders_clean["discount_pct"],
    errors="coerce"
)


In [21]:
orders_clean["discount_pct_clean"] = (
    orders_clean["discount_pct_clean"]
    .fillna(0)
)


In [39]:
invalid_discount = ~orders_clean[
    "discount_pct_clean"
].between(0,1)

In [40]:
invalid_discount.sum()

np.int64(0)

In [41]:
orders_clean.loc[
    invalid_discount,
    "discount_pct_clean"
]=0

In [42]:
orders_clean["shipping_fee_clean"] = pd.to_numeric(
    orders_clean["shipping_fee"],
    errors="coerce"
)

In [43]:
orders_clean["shipping_fee_clean"] = (
    orders_clean["shipping_fee_clean"]
    .fillna(0)
)

In [44]:
orphan_order_rows = orders_clean[
    ~orders_clean["customer_id"].isin(
        customers_clean["customer_id"]
    )
]

In [45]:
orders_clean = orders_clean[
    orders_clean["customer_id"].isin(
        customers_clean["customer_id"]
    )
].copy()

In [136]:
# Clean Order Items

In [22]:
orphan_product_rows = order_items_clean[
    ~order_items_clean["product_id"].isin(
        products_clean["product_id"]
    )
]


In [23]:
order_items_clean = order_items_clean[
    order_items_clean["product_id"].isin(
        products_clean["product_id"]
    )
].copy()


In [48]:
order_items_clean["quantity"] = pd.to_numeric(
    order_items_clean["quantity"],
    errors="coerce"
)

In [49]:
invalid_qty = (
    order_items_clean["quantity"] <= 0
)
invalid_qty.sum()

np.int64(370)

In [50]:
order_items_clean = order_items_clean[
    order_items_clean["quantity"] > 0
].copy()

In [51]:
def clean_money(series):
    return pd.to_numeric(
        series.astype("string")
        .str.replace("₱", "", regex=False)
        .str.replace(",", "", regex=False)
        .str.strip(),
        errors="coerce"
    )

In [52]:
order_items_clean["unit_price_clean"] = clean_money(
    order_items_clean["unit_price_at_purchase"]
)

In [53]:
order_items_clean["unit_price_clean"].isna().sum()

np.int64(2208)

In [54]:
order_items_clean = order_items_clean[
    order_items_clean["unit_price_clean"].notna()
].copy()

In [55]:
order_items_clean["unit_price_clean"].isna().sum()

np.int64(0)

In [ ]:
# Merge Sales Analysis Fact Table

In [179]:
fsales_analysis = order_items_clean.merge(
    orders_clean[
        [
            "order_id",
            "customer_id",
            "order_datetime_clean",
            "order_status",
            "sales_channel",
            "shipping_method",
            "discount_pct_clean",
            "shipping_fee_clean",
        ]
    ],
    on="order_id",
    how="inner",
    validate="many_to_one"
)

In [195]:
fsales_analysis = fsales_analysis.merge(
    products_clean[
        [
            "product_id",
            "product_name",
            "brand",
            "category_clean",
            "unit_cost",
            
        ]
    ],
    on="product_id",
    how="inner",
    validate="many_to_one"
)

In [206]:
fsales_analysis = fsales_analysis.merge(
    customers_clean[
        [
            "customer_id",
            "region_name",
            "city",
            "loyalty_tier",
            "gender_clean",
            "age_clean",
        ]
    ],
    on="customer_id",
    how="left",
    validate="many_to_one"
)

In [212]:
# Calculated Columns in Sales Analysis Fact Tables

In [213]:
fsales_analysis["gross_sales"] = (
    fsales_analysis["quantity"]
    * fsales_analysis["unit_price_clean"]
)

In [214]:
fsales_analysis["net_sales"] = (
    fsales_analysis["gross_sales"]
    * (
        1
        -
        fsales_analysis["discount_pct_clean"]
    )
)

In [215]:
fsales_analysis["order_date"] = (
    fsales_analysis[
        "order_datetime_clean"
    ].dt.date
)

In [217]:
fsales_analysis["month"] = (
    fsales_analysis[
        "order_datetime_clean"
    ]
    .dt.to_period("M")
    .astype("string")
)

In [218]:
fsales_analysis["year"] = (
    fsales_analysis[
        "order_datetime_clean"
    ].dt.year
)

In [219]:
fsales_analysis[
    [
        "order_datetime_clean",
        "order_date",
        "month",
        "year"
    ]
].head()

,order_datetime_clean,order_date,month,year
0,2025-11-07 17:02:00,2025-11-07,2025-11,2025
1,2025-09-18 09:13:00,2025-09-18,2025-09,2025
2,2025-09-18 09:13:00,2025-09-18,2025-09,2025
3,2025-09-18 09:13:00,2025-09-18,2025-09,2025
4,2025-09-18 09:13:00,2025-09-18,2025-09,2025


In [221]:
fsales_analysis["cost_of_sales"] = (
    fsales_analysis["quantity"]
    * fsales_analysis["unit_cost"]
)

In [222]:
fsales_analysis["gross_profit"] = (
    fsales_analysis["gross_sales"]
    - fsales_analysis["cost_of_sales"]
)

In [ ]:
# Export Sales Analyis Fact Table to CSV file

In [227]:
fsales_analysis.to_csv(
    "fsales_analysis.csv",
    index=False
)

In [63]:
customers_clean.to_csv(
    "customers_clean.csv",
    index=False
)

In [65]:
products_clean.to_csv(
    "products_clean.csv",
    index=False
)

In [70]:
orders_clean.to_csv(
    "orders_clean.csv",
    index=False
)

In [69]:
order_items_clean.to_csv(
    "order_items_clean.csv",
    index=False
)